In [1]:
import numpy as np
import pandas as pd

D = 0.008

datos = pd.DataFrame({
    'segmento': ['Constantes', 'Casuales'],
    'precio': [20, 12],
    'costo': [8, 8],
    'unidades': [1000, 2000],
    'eps_x': [0.5, 0.2],
    'churn': [0.03, 0.06]
}).set_index('segmento')

datos

,precio,costo,unidades,eps_x,churn
segmento,,,,,
Constantes,20,8,1000,0.5,0.03
Casuales,12,8,2000,0.2,0.06


In [2]:
pruebas = pd.DataFrame({
    'segmento': ['Constantes'] * 5 + ['Casuales'] * 5,
    'precio': [18, 19, 20, 21, 22,
               10.8, 11.4, 12, 12.6, 13.2],
    'unidades': [1159, 1074, 1000, 934, 875,
                 2603, 2274, 2000, 1770, 1576]
})

pruebas

,segmento,precio,unidades
0,Constantes,18.0,1159
1,Constantes,19.0,1074
2,Constantes,20.0,1000
3,Constantes,21.0,934
4,Constantes,22.0,875
5,Casuales,10.8,2603
6,Casuales,11.4,2274
7,Casuales,12.0,2000
8,Casuales,12.6,1770
9,Casuales,13.2,1576


In [3]:
g = pruebas[pruebas.segmento == 'Constantes']

eps_const = np.polyfit(
    np.log(g.precio),
    np.log(g.unidades),
    1
)[0]

print(f'Elasticidad de Constantes: {eps_const:.2f}')

Elasticidad de Constantes: -1.40


In [5]:
# TU TURNO: elasticidad de Casuales
g = pruebas[pruebas.segmento == 'Casuales']
 
# TU CÓDIGO AQUÍ  (calculen eps_casu con la misma fórmula y escríbanla con print)
eps_casu = np.polyfit(
    np.log(g.precio),
    np.log(g.unidades),
    1
)[0]
print(f'Elasticidad de Casuales: {eps_casu:.2f}')

datos['eps'] = [round(eps_const, 1), round(eps_casu, 1)]   # ε de cada segmento, a un decimal

datos


Elasticidad de Casuales: -2.50


,precio,costo,unidades,eps_x,churn,eps
segmento,,,,,,
Constantes,20,8,1000,0.5,0.03,-1.4
Casuales,12,8,2000,0.2,0.06,-2.5


In [6]:
# Las mismas funciones del Demo 2 de Ritmo (no hay que cambiarlas)
def precio_optimo(C, eps):
    return C * eps / (1 + eps)                 # P* = C × ε ÷ (1 + ε)
 
def aplicar_reglas(p_hoy, p_opt, C):
    p = min(p_opt, p_hoy * 1.10)               # tope: sube máximo 10%
    p = max(p, p_hoy * 0.90)                   # tope: baja máximo 10%
    p = max(p, 1.2 * C)                        # margen mínimo: precio ≥ 1.2 × C
    return p
 
def utilidad(p, p_hoy, C, Q, eps):
    unidades = Q * (p / p_hoy) ** eps          # cuántas unidades se venden a p
    return (p - C) * unidades                  # utilidad = (p − C) × unidades
 
def clv(m, c, d=D):
    return m * (1 - c) / (c + d)               # CLV = m × (1 − c) ÷ (c + d)
 
def churn_maximo(m_nuevo, clv_hoy, d=D):
    return (m_nuevo - clv_hoy * d) / (m_nuevo + clv_hoy)  

In [7]:
# EJEMPLO resuelto: segmento Constantes
s = datos.loc['Constantes']
 
p_opt  = precio_optimo(s.costo, s.eps)
p_prop = aplicar_reglas(s.precio, p_opt, s.costo)
 
print(f'Precio óptimo P*:          ${p_opt:.2f}')
print(f'Precio propuesto (reglas): ${p_prop:.2f}')
print(f'Utilidad hoy:      {utilidad(s.precio, s.precio, s.costo, s.unidades, s.eps):,.0f}')
print(f'Utilidad propuesta:{utilidad(p_prop, s.precio, s.costo, s.unidades, s.eps):,.0f}')

Precio óptimo P*:          $28.00
Precio propuesto (reglas): $22.00
Utilidad hoy:      12,000
Utilidad propuesta:12,251


In [8]:
utilidad_optima = utilidad(28, 20, 8, 1000, -1.4)
utilidad_permitida = utilidad(22, 20, 8, 1000, -1.4)

print(f'Utilidad con $28: ${utilidad_optima:,.2f}')
print(f'Utilidad con $22: ${utilidad_permitida:,.2f}')
print(f'Diferencia: ${utilidad_optima - utilidad_permitida:,.2f}')

Utilidad con $28: $12,486.79
Utilidad con $22: $12,251.19
Diferencia: $235.60


In [10]:
# TU TURNO: repitan lo mismo para el segmento Casuales
s = datos.loc['Casuales']
 
p_opt  = precio_optimo(s.costo, s.eps)
p_prop = aplicar_reglas(s.precio, p_opt, s.costo)
 
print(f'Precio óptimo P*:          ${p_opt:.2f}')
print(f'Precio propuesto (reglas): ${p_prop:.2f}')
print(f'Utilidad hoy:      {utilidad(s.precio, s.precio, s.costo, s.unidades, s.eps):,.0f}')
print(f'Utilidad propuesta:{utilidad(p_prop, s.precio, s.costo, s.unidades, s.eps):,.0f}')

Precio óptimo P*: $13.33
Precio propuesto: $13.20
Utilidad hoy: $8,000
Utilidad propuesta: $8,195


In [12]:
# La función nueva: utilidad de los DOS segmentos juntos, con elasticidad cruzada
def utilidad_total(p_const, p_casu):
    a, b = datos.loc['Constantes'], datos.loc['Casuales']
    q_const = a.unidades * (p_const / a.precio) ** a.eps * (p_casu / b.precio) ** b.eps_x
    q_casu  = b.unidades * (p_casu / b.precio) ** b.eps * (p_const / a.precio) ** a.eps_x
    u_const = (p_const - a.costo) * q_const
    u_casu  = (p_casu - b.costo) * q_casu
    return u_const, u_casu
 
# EJEMPLO resuelto: sube SOLO Constantes a $22 (Casuales se queda en $12)
hoy = utilidad_total(20, 12)
sube = utilidad_total(22, 12)
print(f'Constantes: {hoy[0]:,.0f} -> {sube[0]:,.0f}')
print(f'Casuales:   {hoy[1]:,.0f} -> {sube[1]:,.0f}')
print(f'Total:      {sum(hoy):,.0f} -> {sum(sube):,.0f}')

Constantes: 12,000 -> 12,251
Casuales:   8,000 -> 8,390
Total:      20,000 -> 20,642


In [13]:
# TU TURNO: suben LOS DOS segmentos (Constantes a $22 y Casuales a $13.20)
# TU CÓDIGO AQUÍ  (calculen utilidad_total(22, 13.2) e impriman la utilidad de cada segmento y el total)
hoy = utilidad_total(20, 12)

nueva = utilidad_total(22, 13.2)

print(f'Constantes: ${nueva[0]:,.0f}')
print(f'Casuales: ${nueva[1]:,.0f}')
print(f'Total: ${sum(nueva):,.0f}')

# TU CÓDIGO AQUÍ  (comparen con la suma de las dos utilidades propuestas de las celdas 3 y 4: 12,251 + 8,195 = 20,446)
sin_cruzado = 12251 + 8195

print(f'Sin efecto cruzado: ${sin_cruzado:,.0f}')
print(f'Con efecto cruzado: ${sum(nueva):,.0f}')
print(f'Diferencia: ${sum(nueva) - sin_cruzado:,.0f}')

Constantes: $12,487
Casuales: $8,595
Total: $21,082
Sin efecto cruzado: $20,446
Con efecto cruzado: $21,082
Diferencia: $636


In [14]:
# TU CÓDIGO AQUÍ
# 1) clv_hoy = clv(m, c)           con m = precio de hoy − costo
# 2) m_nuevo = p_prop − costo
# 3) churn_maximo(m_nuevo, clv_hoy)
# Háganlo para los dos segmentos
for nombre in ['Constantes', 'Casuales']:
    s = datos.loc[nombre]
    clv_hoy = clv(s.precio - s.costo, s.churn)
    p_opt = precio_optimo(s.costo, s.eps)
    p_prop = aplicar_reglas(s.precio, p_opt, s.costo)
    m_nuevo = p_prop - s.costo
    c_max = churn_maximo(m_nuevo, clv_hoy)

    print(nombre)
    print(f'CLV actual: ${clv_hoy:.2f}')
    print(f'Churn actual: {s.churn:.2%}')
    print(f'Churn máximo: {c_max:.2%}')
    print()


Constantes
CLV actual: $306.32
Churn actual: 3.00%
Churn máximo: 3.61%

Casuales
CLV actual: $55.29
Churn actual: 6.00%
Churn máximo: 7.86%



In [16]:
escenarios = [0.5, 1, 2.5]

for e in escenarios:
    churn_const = 0.03 * (1 + e * 0.10)
    churn_casu = 0.06 * (1 + e * 0.10)

    print(f'Elasticidad churn: {e}')
    print(f'Constantes: {churn_const:.2%}')
    print(f'Casuales: {churn_casu:.2%}')
    print()


Elasticidad churn: 0.5
Constantes: 3.15%
Casuales: 6.30%

Elasticidad churn: 1
Constantes: 3.30%
Casuales: 6.60%

Elasticidad churn: 2.5
Constantes: 3.75%
Casuales: 7.50%

